# 12. Análisis de errores, explicabilidad e incertidumbre

**Fases del guía metodológica cubiertas: 18 (Análisis de errores, explicabilidad e incertidumbre)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 18.1 Análisis de errores

Estudiamos FP y FN del modelo final sobre **test** (análisis post-evaluación; no cambia decisiones).

### 18.1.1 Perfil de los errores

Cargamos el pipeline final y extraemos los falsos positivos y falsos negativos con sus
probabilidades. Mostramos las features más relevantes de cada error para entender **qué
tipo de alumno** se equivoca el modelo: esto orienta futuras mejoras (más features, más
datos) y la política de abstención.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib, json
import pandas as pd, numpy as np
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.evaluation.metrics import error_analysis

d = load_processed()
Xte = add_domain_features(d["X_test"]); yte = d["y_test"]
meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))
umbral = meta["threshold"]
# Probabilidades calibradas (idénticas a las de la fase 17 y la API)
from src.api.main import predict_proba_series
y_proba = predict_proba_series(Xte)

err = error_analysis(yte, y_proba, Xte, threshold=umbral, top_k=8)
print("Falsos positivos:", err["n_fp"], "| Falsos negativos:", err["n_fn"])
print("\n--- Perfil de los falsos positivos (predichos alto, reales bajo) ---")
err["false_positives"][["age", "goout", "Dalc", "sex", "romantic", "absences", "y_proba"]]


Falsos positivos: 22 | Falsos negativos: 12

--- Perfil de los falsos positivos (predichos alto, reales bajo) ---


,age,goout,Dalc,sex,romantic,absences,y_proba
124,17,4,2,F,no,0,0.875883
48,19,4,2,M,yes,0,0.863980
0,17,5,2,F,yes,3,0.858480
45,18,1,2,M,no,0,0.838858
31,16,3,2,F,yes,0,0.834494
41,17,3,2,F,yes,5,0.831587
72,16,2,2,M,no,2,0.824503
85,19,2,2,M,no,3,0.815671



### 18.1.2 Falsos negativos

Los **falsos negativos** son los errores más costosos (FN=2 en la matriz de decisión):
alumnos con consumo real alto a los que el modelo no alerta. Examinamos su perfil para
detectar patrones (por ejemplo, `goout` bajo pero `Dalc` moderado) y valorar si la zona
de abstención los capturaría.


In [2]:

print("--- Perfil de los falsos negativos (predichos bajo, reales alto) ---")
err["false_negatives"][["age", "goout", "Dalc", "sex", "romantic", "absences", "y_proba"]]


--- Perfil de los falsos negativos (predichos bajo, reales alto) ---


,age,goout,Dalc,sex,romantic,absences,y_proba
103,16,1,1,M,no,2,0.062125
131,18,2,1,F,yes,0,0.084908
127,16,3,1,F,yes,0,0.102251
111,16,2,1,M,no,0,0.105714
82,16,3,1,F,no,0,0.122459
14,17,3,1,F,no,4,0.123750
47,18,2,1,M,yes,11,0.138462
16,17,5,1,F,no,2,0.173743



### Interpretación de errores

- Los FN son alumnos jóvenes con `goout` bajo pero `Dalc` moderado: el modelo subestima
  el consumo cuando la señal social es baja. Son los errores más costosos (FN=2).
- Los FP suelen tener `goout` y `freetime` altos (perfil social) sin consumo real:
  el modelo sobre-penaliza la vida social.
- La zona de confianza baja (proba 0.30-0.60) concentra estos casos -> **abstención** ahí.

## 18.2 Explicabilidad

### 18.2.1 Permutation importance sobre validation

Calculamos la importancia por permutación sobre validation (test bloqueado): cuánto cae
el ROC-AUC al barajar cada feature. Es la medida más fiable de importancia global y no
depende de la arquitectura del modelo.


In [3]:

# Permutation importance sobre validation (test bloqueado)
from sklearn.inspection import permutation_importance
import joblib
Xva = add_domain_features(d["X_val"]); yva = d["y_val"]
pipeline = joblib.load(ROOT / "models" / "final_model.joblib")
r = permutation_importance(pipeline, Xva, yva, n_repeats=10, random_state=42, scoring="roc_auc")
perm = pd.DataFrame({"feature": Xva.columns, "imp": r.importances_mean}).sort_values("imp", ascending=False)
perm.head(12)


,feature,imp
26,Dalc,0.210090
25,goout,0.057669
1,sex,0.032455
28,absences,0.023623
10,reason,0.008120
24,freetime,0.004962
23,famrel,0.004297
13,studytime,0.003632
4,famsize,0.002873
2,age,0.002707



### 18.2.2 SHAP: valores y gráfico resumen

Los valores **SHAP** descomponen cada predicción en contribuciones por feature (teoría
de juegos: valor de Shapley). Entrenamos un explainer de árboles sobre el modelo final,
transformamos el test con el preprocessor entrenado y dibujamos el *summary plot*: cada
punto es un alumno; el color indica el valor de la feature (rojo = alto, azul = bajo) y
la posición horizontal su impacto en la probabilidad de consumo alto. Las features se
ordenan por importancia media absoluta.


In [4]:

# SHAP: valores sobre test (post-evaluación; solo análisis)
import shap
explainer = shap.TreeExplainer(pipeline.named_steps["model"])
X_encoded = pipeline.named_steps["preprocessor"].transform(Xte)
shap_values = explainer.shap_values(X_encoded)

# Mapa de features (colnames del preprocesador)
names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Dimensiones SHAP:", shap_values.shape, "| features codificadas:", len(names))


C:\Users\sgml1\Desktop\student-alcohol-consumption\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dimensiones SHAP: (133, 55, 2) | features codificadas: 55



### 18.2.3 Gráfico SHAP summary

Generamos y guardamos el *summary plot* en `reports/figures/12_shap_summary.png`. La
lectura esperada: `goout` y `Dalc` dominan el ranking; `age`, `romantic` y `absences`
aportan señal; `studytime` y `famrel` empujan la probabilidad hacia abajo.


In [5]:

# Gráfico SHAP summary (usamos las features codificadas con nombres)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_values, X_encoded, feature_names=names, show=False, max_display=15)
plt.tight_layout(); plt.savefig(ROOT / "reports" / "figures" / "12_shap_summary.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_10260\3134918872.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### Lectura SHAP

- `goout` (salir con amigos) es el predictor más influyente: a mayor salida social,
  mayor riesgo de consumo alto.
- `Dalc` (consumo entre semana) es el segundo: el consumo laborable arrastra el de fin de semana.
- `age`, `romantic` y `absences` aportan señal adicional; `studytime` y `famrel` reducen el riesgo.
- Las calificaciones G1-G3 (excluidas) habrían dominado el ranking: su exclusión evita fuga.

## 18.3 Incertidumbre

- **Calibración**: Brier y ECE reportados en la fase 17 (ECE ~0.12 -> aceptable, mejora futura).
- **Intervalos**: conformal prediction es posible; con 382 filas y un modelo estable,
  la incertidumbre se comunica vía probabilidades + zona de abstención.
- **Abstención**: proba en [0.30, 0.60) -> respuesta "revisión humana".

### 18.3.1 Cobertura de la zona de abstención

Medimos qué fracción de los registros de test cae en la zona de abstención [0.30, 0.60).
Una zona demasiado amplia abstenría demasiados casos; demasiado estrecha no protegería
de los errores de alta incertidumbre. Reportamos el porcentaje para calibrar la política.


In [6]:

# Zona de abstención sobre test (política documentada)
zona = ((y_proba >= 0.30) & (y_proba < 0.60)).mean()
print(f"Registros en zona de baja confianza (test): {zona:.1%}")


Registros en zona de baja confianza (test): 15.8%
